# Lecture 5 Activity – Did the South make the Republicans conservative?

QSS 20 · Modern Statistical Computing · Fall 2026

## The question

In lecture we found that the gap between the parties grew slowly until the late
1970s and fast after that, and that most of the later widening was Republicans
moving right.

Over the same years the South, once solidly Democratic, sent more and more
Republicans to the House. Maybe Southern Republicans were more conservative
than other Republicans, and the party's average moved right because it gained
Southern members.

> **Did the Republican party move right because it gained Southern members, or
> did Republicans outside the South move right too?**

### Reminders

- One row is **one member of the House in one Congress**.
- `dwnom1`: negative liberal, positive conservative.
- A mean is always taken over some set of rows. Name that set before you
  interpret the number.
- In the starter cells, replace each `...` with code.

In [1]:
import pandas as pd

pd.set_option('display.max_rows', 8)

congress = pd.read_csv('../data/raw/congress.csv')
house = congress.loc[congress['district'] != 0]

# The eleven states of the former Confederacy, spelled as in the file.
south = ['ALABAMA', 'ARKANSA', 'FLORIDA', 'GEORGIA', 'LOUISIA', 'MISSISS',
         'NORTH C', 'SOUTH C', 'TENNESS', 'TEXAS', 'VIRGINI']

house = house.assign(is_south=house['state'].isin(south))
house = house.assign(region=house['is_south'].map({True: 'South', False: 'Non-South'}))

# Check: the South should have 107 members in the 80th Congress.
house.loc[house['congress'] == 80].groupby('region').size()

region
Non-South    338
South        107
dtype: int64

`region` holds the same information as `is_south`, as the words `South` and
`Non-South`, which makes tables easier to read.

## Task 1 · When did the South flip?

Count the **Southern** members of each party in every Congress, with one row
per Congress and one column per party. Empty cells mean no members, so fill
them with 0.

Then find the first Congress in which Southern Republicans outnumber Southern
Democrats. Before you run it, guess the decade.

In [2]:
seats = (
    house
    .loc[house['is_south']]
    .pivot_table(index='congress', columns='party', values='name',
                 aggfunc='size', fill_value=0)
)
seats

party,Democrat,Other,Republican
congress,,,
80,105,0,2
81,102,0,3
82,103,0,2
83,100,0,6
...,...,...,...
109,49,0,82
110,56,0,79
111,60,0,74
112,37,0,94


In [3]:
seats.loc[seats['Republican'] > seats['Democrat']].head(3)

party,Democrat,Other,Republican
congress,,,
104,61,0,69
105,54,0,71
106,54,1,71


The first Congress with more Southern Republicans than Southern Democrats is
the **104th** (1995–97): 69 against 61. That is the Congress elected in 1994.

The Southern Republican count rose from 2 in the 80th Congress to 48 in the
103rd, but it did not pass the Democratic count until the 104th.

## Task 2 · ❓ Two shares, two denominators

A classmate writes two cells and labels both "Southern Republican share". Run
them.

```python
# A
(house.loc[house['is_south']]
      .assign(is_rep=house['party'] == 'Republican')
      .groupby('congress')['is_rep'].mean())

# B
(house.loc[house['party'] == 'Republican']
      .groupby('congress')['is_south'].mean())
```

1. Over which rows is each mean taken?
2. Which one answers "How Republican was the South?" and which answers "How
   Southern was the Republican party?"

In [4]:
a = (house.loc[house['is_south']]
          .assign(is_rep=house['party'] == 'Republican')
          .groupby('congress')['is_rep'].mean())
b = (house.loc[house['party'] == 'Republican']
          .groupby('congress')['is_south'].mean())

pd.DataFrame({'A': a, 'B': b}).loc[[80, 96, 104, 112]].round(3)

,A,B
congress,,
80,0.019,0.008
96,0.284,0.194
104,0.531,0.292
112,0.718,0.387


**A** is taken over Southern members: the share of them who are Republican, or
how Republican the South was. It rises from 0.019 to 0.718.

**B** is taken over Republicans: the share of them who are Southern, or how
Southern the Republican party was. It rises from 0.008 to 0.387.

## Task 3 · Were Southern Republicans more conservative?

For **Republicans** only, build a table with one row per Congress and region
and two named columns: `members` (row count) and `mean_score` (mean `dwnom1`).

Then reshape `mean_score` so each Congress is one row, with a `South` and a
`Non-South` column, and print the 80th, 95th, 104th and 112th Congresses.

Were Southern Republicans more conservative than other Republicans? Did
**non-Southern** Republicans move?

In [5]:
rep_table = (
    house
    .loc[house['party'] == 'Republican']
    .groupby(['congress', 'region'])
    .agg(members=('name', 'size'), mean_score=('dwnom1', 'mean'))
    .reset_index()
)
rep_table

,congress,region,members,mean_score
0,80,Non-South,248,0.276218
1,80,South,2,0.211500
2,81,Non-South,171,0.264655
3,81,South,3,0.236000
...,...,...,...,...
62,111,Non-South,109,0.647927
63,111,South,74,0.672757
64,112,Non-South,149,0.664893
65,112,South,94,0.691862


In [6]:
rep_wide = rep_table.pivot(index='congress', columns='region', values='mean_score')
rep_wide.loc[[80, 95, 104, 112]].round(3)

region,Non-South,South
congress,,
80,0.276,0.211
95,0.202,0.266
104,0.448,0.502
112,0.665,0.692


From the 84th Congress (1955–57) on, the Southern mean is above the
non-Southern mean in every Congress. In the 112th it was 0.692 against 0.665, a
difference of 0.027.

That difference is small next to how far **non-Southern** Republicans moved:
from 0.202 in the 95th Congress to 0.665 in the 112th. Republicans outside the
South moved right on their own.

## Task 4 · How much did the Southern members add?

If the party moved right **only** because it gained Southern members, then
Republicans outside the South would not have moved at all.

From the 80th to the 112th Congress, compute:

1. the change in the mean `dwnom1` of **all** Republicans;
2. the change in the mean `dwnom1` of **non-Southern** Republicans.

What share of the party's move would still have happened with no Southern
members? Answer today's question in one or two sentences.

In [7]:
all_reps = house.loc[house['party'] == 'Republican'].groupby('congress')['dwnom1'].mean()

party_change = all_reps.loc[112] - all_reps.loc[80]
non_south_change = rep_wide['Non-South'].loc[112] - rep_wide['Non-South'].loc[80]

print('change, all Republicans:         ', round(party_change, 3))
print('change, non-Southern Republicans:', round(non_south_change, 3))
print('share without the South:         ', round(non_south_change / party_change, 2))

change, all Republicans:          0.4
change, non-Southern Republicans: 0.389
share without the South:          0.97


The Republican mean rose by 0.40. Republicans outside the South rose by 0.39 on
their own, so about **97 percent** of the party's move would have happened with
no Southern members.

The party did not move right mainly because it gained Southern members.
Republicans everywhere moved right.

## Task 5 · What the tables cannot show

In two sentences: what do these tables support saying, and what can they not
establish? Think about who is inside "non-Southern Republicans" in 1947 and in
2011.

The tables support a descriptive claim: Republicans outside the South moved
right by almost as much as the party did, so the growing Southern share
explains little of the party's move.

They cannot say **why** non-Southern Republicans moved. The non-Southern
Republicans of 1947 and 2011 are almost entirely different people, so the group
mean mixes members changing how they vote with new members replacing old ones.